# Module 15: Model Training at Scale

**Snowpark ML Fundamentals — Week 4 | Lunch & Learn**

---

## Learning Objectives
- Use **GridSearchCV** and **RandomizedSearchCV** for server-side hyperparameter tuning
- Choose the right scoring metric (accuracy vs. f1 vs. roc_auc) for imbalanced data
- Apply the **coarse-to-fine refinement pattern**: RandomizedSearchCV followed by focused GridSearchCV
- Estimate **warehouse credit costs** for tuning jobs
- Understand **Snowpark-Optimized Warehouses** and when to use them
- Close the loop: search → evaluate on holdout → register → deploy by alias

## Key Concept
> **Hyperparameter tuning in Snowpark ML runs entirely inside the warehouse.**
> The same `input_cols`/`label_cols`/`output_cols` API pattern you already know
> extends to `GridSearchCV`, `RandomizedSearchCV`, and 80+ algorithms — learn
> the pattern once, apply it everywhere.

---

In [1]:
%load_ext autoreload
%autoreload 2

## 15.1 Setup

Create session, generate a churn dataset, preprocess, and split.

In [ ]:
import sys
sys.path.insert(0, '../src')

import logging
import math
import time
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

sns.set_theme(style="whitegrid", palette="colorblind", font_scale=1.1)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data
from snowpark_fundamentals.preprocessing.transformers import build_preprocessing_pipeline
from snowpark_fundamentals.modeling.trainer import train_model, predict, MODEL_REGISTRY
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier, get_confusion_matrix
from snowpark_fundamentals.modeling.tuning import (
    grid_search_cv,
    randomized_search_cv,
    get_search_results,
    get_best_model_params,
)

session = create_session(env_path='../.env')

NUMERIC_COLS = ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'TOTAL_CHARGES',
                'SUPPORT_TICKETS', 'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS']
CATEGORICAL_COLS = ['PLAN_TYPE', 'CONTRACT_TYPE', 'PAYMENT_METHOD']
LABEL_COL = 'CHURNED'
FEATURE_COLS = [f"{c}_SCALED" for c in NUMERIC_COLS] + [f"{c}_ENCODED" for c in CATEGORICAL_COLS]

df = create_customer_churn_dataset(session, n_rows=5000)
df_processed, _ = build_preprocessing_pipeline(df, NUMERIC_COLS, CATEGORICAL_COLS)
train_df, test_df = split_data(df_processed)
print(f"Train: {train_df.count()} rows, Test: {test_df.count()} rows")

# Class distribution — important for metric choice later
churn_counts = df.group_by(LABEL_COL).count().to_pandas()
print(f"\nClass distribution:\n{churn_counts.to_string(index=False)}")
churn_rate = churn_counts[churn_counts[LABEL_COL] == 1]['COUNT'].values[0] / churn_counts['COUNT'].sum()
print(f"Churn rate: {churn_rate:.1%} — this is imbalanced data")

## 15.2 Snowpark-Optimized Warehouses

Standard warehouses allocate memory proportional to their credit cost. **Snowpark-Optimized Warehouses** provide **16× more memory per node**, designed for ML training and large-scale data processing.

| Size | Standard Memory | Snowpark-Optimized Memory |
|------|----------------|---------------------------|
| XS   | 1 GB           | 16 GB                     |
| S    | 2 GB           | 32 GB                     |
| M    | 16 GB          | 256 GB                    |
| L    | 64 GB          | 1 TB                      |

**When to use Snowpark-Optimized:**
- Hyperparameter searches with many CV folds
- Training on datasets that exceed standard warehouse memory
- Gradient boosting with deep trees or many estimators

```sql
-- Create a dedicated ML training warehouse (requires ACCOUNTADMIN or CREATE WAREHOUSE privilege)
CREATE WAREHOUSE IF NOT EXISTS ML_TRAINING_WH
    WAREHOUSE_TYPE = 'SNOWPARK-OPTIMIZED'
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 300
    AUTO_RESUME = TRUE;
```

> **Best practice:** Use a separate warehouse for ML training so queries and training don't compete for resources. Ref: [Snowpark-Optimized Warehouses](https://docs.snowflake.com/en/user-guide/warehouses-snowpark-optimized)

### Credit Cost Estimation

Snowflake charges per-second with a 60-second minimum. The credit rate depends on warehouse size and type:

| Size | Standard (credits/hr) | Snowpark-Optimized (credits/hr) |
|------|----------------------|--------------------------------|
| XS   | 1                    | 1.5                            |
| S    | 2                    | 3                              |
| M    | 8                    | 12                             |
| L    | 16                   | 24                             |

**Cost formula:** `credits = (credits_per_hour) × (wall_time_seconds / 3600)`

A 12-combination GridSearchCV that takes 90 seconds on a Medium Snowpark-Optimized warehouse costs `12 × (90 / 3600) = 0.30 credits`. We will compute this for every search in this notebook.

In [ ]:
# Current warehouse info
current_wh = session.get_current_warehouse()
print(f"Current warehouse: {current_wh}")

# SQL to create a Snowpark-Optimized warehouse (for reference — not executed here)
create_wh_sql = """
CREATE WAREHOUSE IF NOT EXISTS ML_TRAINING_WH
    WAREHOUSE_TYPE = 'SNOWPARK-OPTIMIZED'
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 300
    AUTO_RESUME = TRUE;
"""
print("\nReference SQL for production:")
print(create_wh_sql)

# Credit rate constant — adjust for your warehouse size/type
CREDITS_PER_HOUR = 8  # Medium Standard warehouse
print(f"Using {CREDITS_PER_HOUR} credits/hr for cost estimates (Medium Standard WH)")

## 15.3 GridSearchCV: Exhaustive Hyperparameter Search

Snowpark ML's `GridSearchCV` mirrors scikit-learn's API but executes cross-validation **server-side** on the warehouse. The key difference is the column-based API:

| | scikit-learn | Snowpark ML |
|---|---|---|
| Input | `X` (numpy array) | `input_cols` (column names) |
| Labels | `y` (numpy array) | `label_cols` (column names) |
| Output | `predict(X)` returns array | `output_cols` (column names) |
| Execution | Local CPU/RAM | Snowflake warehouse |
| Data movement | All data in memory | Data stays in Snowflake |

Grid search evaluates **every combination** in the parameter grid. For a grid with 2×3×2 = 12 combinations and 5-fold CV, that's 60 model fits.

> Ref: [GridSearchCV API](https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/api/modeling/snowflake.ml.modeling.model_selection.GridSearchCV)

In [ ]:
# Define a small parameter grid: 2 x 3 x 2 = 12 combinations
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
}

print(f"Grid search: {2 * 3 * 2} combinations x 5 folds = {2 * 3 * 2 * 5} fits")

t0 = time.time()
grid_search = grid_search_cv(
    train_df,
    feature_cols=FEATURE_COLS,
    label_col=LABEL_COL,
    model_type='xgboost',
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
)
grid_time = time.time() - t0

grid_best = get_best_model_params(grid_search)
print(f"\nBest params: {grid_best['best_params']}")
print(f"Best CV accuracy: {grid_best['best_score']:.4f}")
print(f"Candidates evaluated: {grid_best['n_candidates_evaluated']}")
print(f"Wall time: {grid_time:.1f}s")
print(f"Estimated credits: {CREDITS_PER_HOUR * grid_time / 3600:.3f}")

In [ ]:
# Visualize grid search results as a heatmap
grid_results = get_search_results(grid_search)
grid_results_df = pd.DataFrame(grid_results)

# Flatten params dict into columns for plotting
for key in grid_results[0]['params']:
    grid_results_df[key] = grid_results_df['params'].apply(lambda p, k=key: p[k])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for i, n_est in enumerate(sorted(grid_results_df['n_estimators'].unique())):
    subset = grid_results_df[grid_results_df['n_estimators'] == n_est]
    pivot = subset.pivot(index='max_depth', columns='learning_rate', values='mean_test_score')
    sns.heatmap(
        pivot, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[i],
        vmin=grid_results_df['mean_test_score'].min(),
        vmax=grid_results_df['mean_test_score'].max(),
    )
    axes[i].set_title(f'n_estimators = {n_est}')
fig.suptitle('GridSearchCV: Mean CV Accuracy by Hyperparameter Combination', y=1.02)
plt.tight_layout()
plt.show()

### Scoring Metric Matters: Accuracy vs. F1

Our churn dataset is ~78% non-churn / ~22% churn. A model that predicts "no churn" for every customer achieves **78% accuracy** — and catches **zero** actual churners.

| Metric | What it measures | When to use |
|--------|-----------------|-------------|
| `accuracy` | Overall correct predictions | Balanced classes |
| `f1` | Harmonic mean of precision & recall | Imbalanced classes (default choice) |
| `roc_auc` | Ranking quality across thresholds | When you need probability ordering |

**Rule of thumb:** If the minority class is <30% of the data, use `f1` or `roc_auc` as your CV scoring metric, not `accuracy`.

Let's re-run the same grid with `scoring='f1'` and compare.

In [ ]:
# Same grid, different scoring metric
t0 = time.time()
grid_search_f1 = grid_search_cv(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost',
    param_grid=param_grid,
    cv=5,
    scoring='f1',
)
grid_f1_time = time.time() - t0
grid_f1_best = get_best_model_params(grid_search_f1)

# Compare: did the best params change?
print("scoring='accuracy' best params:", grid_best['best_params'])
print(f"  Best CV accuracy: {grid_best['best_score']:.4f}")
print()
print("scoring='f1' best params:      ", grid_f1_best['best_params'])
print(f"  Best CV F1:       {grid_f1_best['best_score']:.4f}")
print()
if grid_best['best_params'] != grid_f1_best['best_params']:
    print("The best hyperparameters CHANGED — the metric you optimize shapes the model you get.")
else:
    print("Same params won under both metrics (uncommon with imbalanced data).")

## 15.4 RandomizedSearchCV: Scaling to Large Search Spaces

> **Key insight:** When you have many hyperparameters, grid search becomes intractable.
> 5 parameters × 5 values each = **3,125 combinations**. Randomized search samples
> a fixed budget of combinations, often finding near-optimal results with far fewer fits.

| Strategy | 5 params × 5 values | 5-fold CV | Total fits |
|----------|--------------------|-----------|-----------|
| Grid     | 3,125 combos       | 5         | **15,625** |
| Random (n_iter=20) | 20 combos | 5         | **100**    |

> Ref: [RandomizedSearchCV API](https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/api/modeling/snowflake.ml.modeling.model_selection.RandomizedSearchCV)

In [ ]:
# Larger search space: 5 parameters with wide ranges
param_distributions = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [3, 4, 5, 6, 8, 10],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
}

total_grid = math.prod(len(v) for v in param_distributions.values())
n_iter = 20
print(f"Full grid would be {total_grid} combos. Sampling {n_iter} instead.")

t0 = time.time()
random_search = randomized_search_cv(
    train_df,
    feature_cols=FEATURE_COLS,
    label_col=LABEL_COL,
    model_type='xgboost',
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=5,
    scoring='f1',
    random_state=42,
)
random_time = time.time() - t0

random_best = get_best_model_params(random_search)
print(f"\nBest params: {random_best['best_params']}")
print(f"Best CV F1: {random_best['best_score']:.4f}")
print(f"Candidates evaluated: {random_best['n_candidates_evaluated']}")
print(f"Wall time: {random_time:.1f}s")
print(f"Estimated credits: {CREDITS_PER_HOUR * random_time / 3600:.3f}")

In [ ]:
# Visualize all 20 candidates with error bars
random_results = get_search_results(random_search)
random_results_df = pd.DataFrame(random_results)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('colorblind', len(random_results_df))
ax.barh(
    range(len(random_results_df)),
    random_results_df['mean_test_score'],
    xerr=random_results_df['std_test_score'],
    color=colors,
    edgecolor='black', linewidth=0.5,
)
ax.set_yticks(range(len(random_results_df)))
ax.set_yticklabels([f"Rank {r}" for r in random_results_df['rank_test_score']])
ax.set_xlabel('Mean CV F1 Score')
ax.set_title('RandomizedSearchCV: All 20 Candidates Ranked')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 3 configurations:")
for _, row in random_results_df.head(3).iterrows():
    print(f"  Rank {row['rank_test_score']}: F1={row['mean_test_score']:.4f} "
          f"(+/- {row['std_test_score']:.4f}) — {row['params']}")

## 15.5 Coarse-to-Fine Refinement: The Production Pattern

The most effective hyperparameter tuning strategy is **two stages**:

1. **Coarse search** (RandomizedSearchCV, n_iter=20–50): Explore a wide space cheaply. Identify the *region* that works.
2. **Fine search** (GridSearchCV around the winner): Build a tight grid centered on the best parameters from stage 1. Exhaustively search the neighborhood.

This is strictly better than running either search alone:
- Cheaper than a full grid over the wide space
- More thorough than random search alone in the winning region

```
Wide Random Search (20 samples from 3,750 combos)
       │
       ▼
Best region identified
       │
       ▼
Focused Grid Search (neighbors of best params)
       │
       ▼
Final best model
```

In [ ]:
# Stage 2: Build a focused grid around the randomized search winner
best = random_best['best_params']


def neighborhood(values_list, best_val):
    """Return the best value and its immediate neighbors from the original list."""
    sorted_vals = sorted(values_list)
    idx = sorted_vals.index(best_val) if best_val in sorted_vals else 0
    lo = max(0, idx - 1)
    hi = min(len(sorted_vals), idx + 2)
    return sorted_vals[lo:hi]


focused_grid = {
    key: neighborhood(param_distributions[key], best[key])
    for key in param_distributions
}

n_focused = math.prod(len(v) for v in focused_grid.values())
print(f"Focused grid: {n_focused} combinations (vs. {total_grid} full grid)")
print(f"Reduction: {n_focused / total_grid:.1%} of the full grid\n")
for k, v in focused_grid.items():
    print(f"  {k}: {v}  (best was {best[k]})")

In [ ]:
# Run the focused grid search
t0 = time.time()
focused_search = grid_search_cv(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost',
    param_grid=focused_grid,
    cv=5,
    scoring='f1',
)
focused_time = time.time() - t0
focused_best = get_best_model_params(focused_search)

print(f"Focused grid best CV F1: {focused_best['best_score']:.4f}")
print(f"Focused grid best params: {focused_best['best_params']}")
print(f"Wall time: {focused_time:.1f}s | Credits: {CREDITS_PER_HOUR * focused_time / 3600:.3f}")

In [ ]:
# Three-strategy comparison
comparison = pd.DataFrame([
    {
        'Strategy': 'GridSearchCV (accuracy)',
        'Candidates': grid_best['n_candidates_evaluated'],
        'Best Score': f"{grid_best['best_score']:.4f}",
        'Metric': 'accuracy',
        'Wall Time (s)': f"{grid_time:.1f}",
        'Est. Credits': f"{CREDITS_PER_HOUR * grid_time / 3600:.3f}",
    },
    {
        'Strategy': 'RandomizedSearchCV (f1)',
        'Candidates': random_best['n_candidates_evaluated'],
        'Best Score': f"{random_best['best_score']:.4f}",
        'Metric': 'f1',
        'Wall Time (s)': f"{random_time:.1f}",
        'Est. Credits': f"{CREDITS_PER_HOUR * random_time / 3600:.3f}",
    },
    {
        'Strategy': 'Coarse\u2192Fine (f1)',
        'Candidates': f"{random_best['n_candidates_evaluated']}+{focused_best['n_candidates_evaluated']}",
        'Best Score': f"{focused_best['best_score']:.4f}",
        'Metric': 'f1',
        'Wall Time (s)': f"{random_time + focused_time:.1f}",
        'Est. Credits': f"{CREDITS_PER_HOUR * (random_time + focused_time) / 3600:.3f}",
    },
])
print(comparison.to_string(index=False))

# Side-by-side bar charts: score vs cost
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
strategies = ['Grid\n(accuracy)', 'Random\n(f1)', 'Coarse\u2192Fine\n(f1)']
scores = [grid_best['best_score'], random_best['best_score'], focused_best['best_score']]
credits = [
    CREDITS_PER_HOUR * grid_time / 3600,
    CREDITS_PER_HOUR * random_time / 3600,
    CREDITS_PER_HOUR * (random_time + focused_time) / 3600,
]

colors = sns.color_palette('colorblind', 3)
ax1.bar(strategies, scores, color=colors, edgecolor='black')
ax1.set_ylabel('Best CV Score')
ax1.set_title('Score by Strategy')
ax1.set_ylim(min(scores) - 0.02, max(scores) + 0.02)

ax2.bar(strategies, credits, color=colors, edgecolor='black')
ax2.set_ylabel('Estimated Credits')
ax2.set_title('Cost by Strategy')

fig.suptitle('Tuning Strategy Comparison: Quality vs. Cost', y=1.02)
plt.tight_layout()
plt.show()

## 15.6 Beyond the Big Three: Snowpark ML Algorithm Library

So far we've used XGBoost, RandomForest, and LogisticRegression. Snowpark ML wraps **80+ algorithms** from scikit-learn and XGBoost, all with the same `input_cols`/`label_cols`/`output_cols` API:

| Family | Module | Example Classes |
|--------|--------|----------------|
| Ensemble | `snowflake.ml.modeling.ensemble` | GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier |
| Linear | `snowflake.ml.modeling.linear_model` | SGDClassifier, RidgeClassifier, Perceptron |
| Trees | `snowflake.ml.modeling.tree` | DecisionTreeClassifier, ExtraTreeClassifier |
| SVM | `snowflake.ml.modeling.svm` | SVC, LinearSVC, NuSVC |
| Neighbors | `snowflake.ml.modeling.neighbors` | KNeighborsClassifier |
| XGBoost | `snowflake.ml.modeling.xgboost` | XGBClassifier, XGBRegressor |

> **The API is the insight, not the algorithm.** Once you know the `input_cols`/`label_cols`/`output_cols` pattern, every algorithm is one import away. Ref: [Snowpark ML Modeling](https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/modeling)

In [ ]:
from snowflake.ml.modeling.ensemble import GradientBoostingClassifier

# Temporarily extend MODEL_REGISTRY so train_model() works with GradientBoosting
MODEL_REGISTRY['gradient_boosting'] = GradientBoostingClassifier

# Train with the same API pattern
model_gb = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='gradient_boosting',
    model_params={'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1},
)
preds_gb = predict(model_gb, test_df)
metrics_gb = evaluate_binary_classifier(preds_gb, LABEL_COL, 'PREDICTION')

# Compare against XGBoost baseline
model_xgb = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost',
    model_params={'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1},
)
preds_xgb = predict(model_xgb, test_df)
metrics_xgb = evaluate_binary_classifier(preds_xgb, LABEL_COL, 'PREDICTION')

# Grouped bar chart
algo_comparison = pd.DataFrame([
    {'Algorithm': 'XGBClassifier', **metrics_xgb},
    {'Algorithm': 'GradientBoostingClassifier', **metrics_gb},
])

fig, ax = plt.subplots(figsize=(8, 4))
algo_comparison.set_index('Algorithm').plot(kind='bar', ax=ax, edgecolor='black')
ax.set_ylabel('Score')
ax.set_title('Algorithm Comparison (Same Hyperparameters)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Clean up the temporary registry entry
del MODEL_REGISTRY['gradient_boosting']

## 15.7 From Search to Registry: Closing the Experiment Loop

Finding the best hyperparameters is only half the job. The complete workflow:
**search → evaluate on holdout → register with metadata → alias for deployment.**

In [ ]:
# Extract the best model from the coarse-to-fine search and evaluate on holdout
best_model = focused_search.best_estimator_
preds_best = best_model.predict(test_df)
metrics_best = evaluate_binary_classifier(preds_best, LABEL_COL, 'PREDICTION')

print("Best model holdout metrics:")
for metric, value in metrics_best.items():
    print(f"  {metric}: {value:.4f}")

# Confusion matrix visualization
cm = get_confusion_matrix(preds_best, LABEL_COL, 'PREDICTION')
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Not Churned', 'Churned'],
    yticklabels=['Not Churned', 'Churned'],
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Holdout Set Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
from snowpark_fundamentals.registry.model_registry import (
    get_registry, log_model, delete_model, set_model_alias, set_model_comment,
)

current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

# Clean up from previous runs
try:
    delete_model(registry, 'CHURN_TUNED_W4')
    print("Deleted existing CHURN_TUNED_W4 (re-run cleanup)")
except Exception:
    pass

sample_input = test_df.select(FEATURE_COLS).limit(10)

# Register the best model from the coarse-to-fine search
mv = log_model(
    registry=registry,
    model=best_model.to_xgboost(),
    model_name='CHURN_TUNED_W4',
    version_name='V1_COARSE_TO_FINE',
    sample_input=sample_input,
    metrics=metrics_best,
)
print("Registered CHURN_TUNED_W4 V1_COARSE_TO_FINE")

# Set production alias
set_model_alias(registry, 'CHURN_TUNED_W4', 'V1_COARSE_TO_FINE', 'production')
print("Set 'production' alias")

# Log full provenance: strategy + metric + cost
search_config = (
    f"Coarse-to-Fine | RandomizedSearchCV(n_iter=20) \u2192 "
    f"GridSearchCV({focused_best['n_candidates_evaluated']} combos) | "
    f"scoring=f1 | best_cv_f1={focused_best['best_score']:.4f} | "
    f"total_credits={CREDITS_PER_HOUR * (random_time + focused_time) / 3600:.3f}"
)
set_model_comment(registry, 'CHURN_TUNED_W4', search_config, version_name='V1_COARSE_TO_FINE')
print(f"Comment: {search_config}")

In [ ]:
from snowpark_fundamentals.registry.model_registry import list_versions, get_model_metrics

# Verify registration
versions_df = list_versions(registry, 'CHURN_TUNED_W4')
print("Registered versions:")
print(versions_df[['name', 'aliases', 'is_default_version']].to_string())

print("\nStored metrics:")
stored_metrics = get_model_metrics(registry, 'CHURN_TUNED_W4', 'V1_COARSE_TO_FINE')
for k, v in stored_metrics.items():
    print(f"  {k}: {v}")

# Validate that metrics were stored correctly
assert stored_metrics['accuracy'] == metrics_best['accuracy'], "Accuracy mismatch!"
print("\nValidation passed.")

## 15.8 Production Recommendations

| # | Recommendation | Details |
|---|---|---|
| 1 | **Use coarse-to-fine tuning** | RandomizedSearchCV(n=20–50) → focused GridSearchCV around the winner |
| 2 | **Score with f1 or roc_auc for imbalanced data** | Accuracy misleads when minority class is <30% |
| 3 | **Estimate credits before launching** | `credits = credits_per_hour × wall_seconds / 3600` |
| 4 | **Match warehouse to workload** | Standard for dev, Snowpark-Optimized for production training |
| 5 | **Log search provenance with the model** | Strategy, scoring metric, n_iter, CV score, and credit cost |
| 6 | **Use `n_jobs=-1`** | Maximizes parallelism across warehouse nodes |
| 7 | **Always evaluate on a holdout set** | CV score estimates generalization; holdout confirms it |

### End-to-End Workflow

```
Define Wide Search Space
       │
       ▼
Coarse Search  (RandomizedSearchCV, scoring='f1')
       │
       ▼
Build Focused Grid  (neighbors of best params)
       │
       ▼
Fine Search  (GridSearchCV, scoring='f1')
       │
       ▼
Holdout Evaluation  (evaluate_binary_classifier)
       │
       ▼
Register + Log Provenance  (log_model + set_model_comment)
       │
       ▼
Deploy by Alias  (set_model_alias → 'production')
```

## Key Takeaways

1. **Coarse-to-fine** is the production tuning pattern: RandomizedSearchCV to explore, focused GridSearchCV to exploit
2. **Scoring metric shapes your model**: use f1 or roc_auc for imbalanced data, never accuracy alone
3. **Every search has a credit cost**: estimate before launching, log after completion
4. **Snowpark-Optimized Warehouses** provide 16× memory per node — essential for large-scale training
5. The **unified `input_cols`/`label_cols`/`output_cols` API** extends across 80+ algorithms and both search strategies
6. The complete workflow is **coarse search → fine search → holdout → registry → alias**

---

**End of Week 4 — Model Training at Scale**

## Cleanup

In [ ]:
# Optional: delete the model registered in this notebook
# delete_model(registry, 'CHURN_TUNED_W4')

session.close()